<a href="https://colab.research.google.com/github/Itishree91/Lang_translation/blob/main/NLP_Er_FR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets sacrebleu sentencepiece torch accelerate evaluate

In [2]:
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    MarianMTModel,
    MarianTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:
# (English, French) sentence pairs
pairs = [
    ("Good morning, how are you?", "Bonjour, comment allez-vous ?"),
    ("I would like a cup of coffee.", "Je voudrais une tasse de café."),
    ("Where is the nearest train station?", "Où est la gare la plus proche ?"),
    ("The weather is beautiful today.", "Il fait beau aujourd'hui."),
    ("Can you help me with this problem?", "Pouvez-vous m'aider avec ce problème ?"),
    ("I am learning to speak French.", "J'apprends à parler français."),
    ("She works as a software engineer.", "Elle travaille comme ingénieure logicielle."),
    ("What time does the museum open?", "À quelle heure ouvre le musée ?"),
    ("This restaurant serves excellent food.", "Ce restaurant sert une excellente nourriture."),
    ("I need to book a hotel room.", "Je dois réserver une chambre d'hôtel."),
    ("The meeting has been rescheduled.", "La réunion a été reportée."),
    ("Thank you very much for your help.", "Merci beaucoup pour votre aide."),
    ("He is reading a very interesting book.", "Il lit un livre très intéressant."),
    ("Please send me the report by tomorrow.", "Veuillez m'envoyer le rapport d'ici demain."),
    ("The children are playing in the park.", "Les enfants jouent dans le parc."),
    ("I lost my passport at the airport.", "J'ai perdu mon passeport à l'aéroport."),
    ("We should leave early to avoid traffic.", "Nous devrions partir tôt pour éviter les embouteillages."),
    ("This is the best pizza I have ever eaten.", "C'est la meilleure pizza que j'aie jamais mangée."),
    ("The company launched a new product.", "L'entreprise a lancé un nouveau produit."),
    ("Could you repeat that, please?", "Pourriez-vous répéter cela, s'il vous plaît ?"),
]

en_texts = [p[0] for p in pairs]
fr_texts = [p[1] for p in pairs]

n = len(pairs)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

def build_dataset(en_list, fr_list):
    return Dataset.from_dict({"en": en_list, "fr": fr_list})

raw_datasets = DatasetDict({
    "train": build_dataset(en_texts[:train_end], fr_texts[:train_end]),
    "validation": build_dataset(en_texts[train_end:val_end], fr_texts[train_end:val_end]),
    "test": build_dataset(en_texts[val_end:], fr_texts[val_end:]),
})

raw_datasets

DatasetDict({
    train: Dataset({
        features: ['en', 'fr'],
        num_rows: 14
    })
    validation: Dataset({
        features: ['en', 'fr'],
        num_rows: 3
    })
    test: Dataset({
        features: ['en', 'fr'],
        num_rows: 3
    })
})

In [4]:
bleu_metric = evaluate.load("sacrebleu")
MAX_LEN = 64

def preprocess_fn(examples, tokenizer, src_lang, tgt_lang):
    inputs = examples[src_lang]
    targets = examples[tgt_lang]
    model_inputs = tokenizer(inputs, max_length=MAX_LEN, truncation=True)
    labels = tokenizer(text_target=targets, max_length=MAX_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

def compute_metrics_fn(tokenizer):
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        decoded_labels = [[l] for l in decoded_labels]
        result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
        return {"bleu": result["score"]}
    return compute_metrics

def train_translation_model(base_checkpoint, src_lang, tgt_lang, output_dir, epochs=3):
    print(f"\n=== Training {src_lang} -> {tgt_lang} (base: {base_checkpoint}) ===")
    tokenizer = MarianTokenizer.from_pretrained(base_checkpoint)
    model = MarianMTModel.from_pretrained(base_checkpoint).to(device)

    tokenized = raw_datasets.map(
        lambda ex: preprocess_fn(ex, tokenizer, src_lang, tgt_lang),
        batched=True,
        remove_columns=raw_datasets["train"].column_names,
    )

    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=2e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        num_train_epochs=epochs,
        predict_with_generate=True,
        logging_steps=2,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        data_collator=data_collator,
        processing_class=tokenizer,
        compute_metrics=compute_metrics_fn(tokenizer),
    )

    trainer.train()
    return trainer, model, tokenizer, tokenized

In [5]:
from google.colab import userdata
userdata.get('Google_Colab')
en_fr_trainer, en_fr_model, en_fr_tokenizer, en_fr_tokenized = train_translation_model(
    base_checkpoint="Helsinki-NLP/opus-mt-en-fr",
    src_lang="en",
    tgt_lang="fr",
    output_dir="./en-fr-finetuned",
    epochs=3,
)


=== Training en -> fr (base: Helsinki-NLP/opus-mt-en-fr) ===


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

...

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Bleu
1,0.343434,0.279252,69.775676
2,0.257706,0.265957,69.775676
3,0.170475,0.261828,69.775676


In [6]:
fr_en_trainer, fr_en_model, fr_en_tokenizer, fr_en_tokenized = train_translation_model(
    base_checkpoint="Helsinki-NLP/opus-mt-fr-en",
    src_lang="fr",
    tgt_lang="en",
    output_dir="./fr-en-finetuned",
    epochs=3,
)


=== Training fr -> en (base: Helsinki-NLP/opus-mt-fr-en) ===


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

...

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Bleu
1,0.422350,0.483250,75.172201
2,0.464475,0.465417,75.172201
3,0.236002,0.458220,75.172201


In [7]:
print("EN -> FR test set results:")
en_fr_test_results = en_fr_trainer.evaluate(en_fr_tokenized["test"])
print(en_fr_test_results)

print("\nFR -> EN test set results:")
fr_en_test_results = fr_en_trainer.evaluate(fr_en_tokenized["test"])
print(fr_en_test_results)

EN -> FR test set results:


Training Loss,Validation Loss,Epoch,Bleu
0.170475,0.265297,3,75.883392


{'eval_loss': 0.26529738306999207, 'eval_bleu': 75.88339225169338}

FR -> EN test set results:


Training Loss,Validation Loss,Epoch,Bleu
0.236002,0.344452,3,40.363627


{'eval_loss': 0.3444521129131317, 'eval_bleu': 40.3636267270949}


In [8]:
def translate(text, model, tokenizer, max_len=64):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len).to(device)
    with torch.no_grad():
        generated = model.generate(**inputs, max_length=max_len, num_beams=4)
    return tokenizer.decode(generated[0], skip_special_tokens=True)

def translate_en_to_fr(text):
    return translate(text, en_fr_model, en_fr_tokenizer)

def translate_fr_to_en(text):
    return translate(text, fr_en_model, fr_en_tokenizer)

In [9]:
# English -> French examples
test_en_sentences = [
    "Hello, how are you doing today?",
    "I would like to order a coffee.",
    "The train leaves in ten minutes."
]

for s in test_en_sentences:
    print(f"EN: {s}")
    print(f"FR: {translate_en_to_fr(s)}")
    print()

EN: Hello, how are you doing today?
FR: Bonjour, comment allez-vous aujourd'hui ?

EN: I would like to order a coffee.
FR: Je voudrais commander un café.

EN: The train leaves in ten minutes.
FR: Le train part dans dix minutes.



In [10]:
# French -> English examples
test_fr_sentences = [
    "Bonjour, comment ça va aujourd'hui ?",
    "Je voudrais commander un café.",
    "Le train part dans dix minutes."
]

for s in test_fr_sentences:
    print(f"FR: {s}")
    print(f"EN: {translate_fr_to_en(s)}")
    print()

FR: Bonjour, comment ça va aujourd'hui ?
EN: Hello, how are you today?

FR: Je voudrais commander un café.
EN: I'd like to order a coffee.

FR: Le train part dans dix minutes.
EN: The train leaves in ten minutes.



In [11]:
en_fr_model.save_pretrained("./en-fr-finetuned/final")
en_fr_tokenizer.save_pretrained("./en-fr-finetuned/final")

fr_en_model.save_pretrained("./fr-en-finetuned/final")
fr_en_tokenizer.save_pretrained("./fr-en-finetuned/final")

print("Models saved to ./en-fr-finetuned/final and ./fr-en-finetuned/final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Models saved to ./en-fr-finetuned/final and ./fr-en-finetuned/final
